# Heat Science Validation Notebook

This notebook validates the project's heat-risk classification
and project-defined overnight recovery indicator against a shared
test-vector table.

The same test vectors should be used by both:
- Python implementation
- TypeScript implementation

so both implementations produce identical results.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

# Setup path for shared test vectors
RESEARCH_DIR = Path("../")
TEST_VECTORS_DIR = RESEARCH_DIR / "test-vectors"
TEST_VECTORS_DIR.mkdir(parents=True, exist_ok=True)
CSV_PATH = TEST_VECTORS_DIR / "heat_science_test_vectors.csv"

print(f"Setup complete. Target CSV path: {CSV_PATH.resolve()}")

Setup complete. Target CSV path: C:\Users\nasse\temperature-api-quickstart\test-vectors\heat_science_test_vectors.csv


In [10]:
test_vectors_data = [
    {
        "test_id": "TC-001",
        "heat_index_f": 78.0,
        "hours_below_80f": 7.0,
        "expected_nws_category": "Below Caution",
        "expected_recovery_indicator": True,
    },
    {
        "test_id": "TC-002",
        "heat_index_f": 85.0,
        "hours_below_80f": 5.0,
        "expected_nws_category": "Caution",
        "expected_recovery_indicator": False,
    },
    {
        "test_id": "TC-003",
        "heat_index_f": 98.0,
        "hours_below_80f": 4.0,
        "expected_nws_category": "Extreme Caution",
        "expected_recovery_indicator": False,
    },
    {
        "test_id": "TC-004",
        "heat_index_f": 110.0,
        "hours_below_80f": 2.0,
        "expected_nws_category": "Danger",
        "expected_recovery_indicator": False,
    },
    {
        "test_id": "TC-005",
        "heat_index_f": 128.0,
        "hours_below_80f": 0.0,
        "expected_nws_category": "Extreme Danger",
        "expected_recovery_indicator": False,
    },
]

df_vectors = pd.DataFrame(test_vectors_data)

display(df_vectors)

,test_id,heat_index_f,hours_below_80f,expected_nws_category,expected_recovery_indicator
0,TC-001,78.0,7.0,Below Caution,True
1,TC-002,85.0,5.0,Caution,False
2,TC-003,98.0,4.0,Extreme Caution,False
3,TC-004,110.0,2.0,Danger,False
4,TC-005,128.0,0.0,Extreme Danger,False


In [11]:
# NWS Heat Index screening bands used by the project.
# The <80°F label is project-defined and is not an official NWS category.

CAUTION_MIN_F = 80.0
EXTREME_CAUTION_MIN_F = 90.0
DANGER_MIN_F = 103.0
EXTREME_DANGER_MIN_F = 125.0

# Project-defined overnight recovery indicator.
RECOVERY_THRESHOLD_F = 80.0
RECOVERY_MIN_HOURS = 6.0

In [12]:
def classify_nws_heat_index(heat_index_f: float) -> str:
    """
    Classify Heat Index using the project-approved screening bands.

    Note:
    - "Below Caution" is a project-defined label for values below 80°F.
    - The four named NWS screening bands begin at 80°F.
    """
    if pd.isna(heat_index_f):
        raise ValueError("heat_index_f cannot be missing.")

    heat_index_f = float(heat_index_f)

    if heat_index_f < CAUTION_MIN_F:
        return "Below Caution"
    elif heat_index_f < EXTREME_CAUTION_MIN_F:
        return "Caution"
    elif heat_index_f < DANGER_MIN_F:
        return "Extreme Caution"
    elif heat_index_f < EXTREME_DANGER_MIN_F:
        return "Danger"
    else:
        return "Extreme Danger"

In [13]:
def evaluate_project_overnight_recovery(hours_below_80f: float) -> bool:
    """
    Project-defined overnight recovery indicator.

    Returns True when conditions remain below the project's
    80°F recovery threshold for at least 6 hours.

    This is a project-defined indicator and is NOT presented
    as an OSHA/NIOSH standard.
    """
    if pd.isna(hours_below_80f):
        raise ValueError("hours_below_80f cannot be missing.")

    hours_below_80f = float(hours_below_80f)

    if hours_below_80f < 0:
        raise ValueError("hours_below_80f cannot be negative.")

    return hours_below_80f >= RECOVERY_MIN_HOURS

In [14]:
assert classify_nws_heat_index(79.9) == "Below Caution"
assert classify_nws_heat_index(80.0) == "Caution"
assert classify_nws_heat_index(89.9) == "Caution"
assert classify_nws_heat_index(90.0) == "Extreme Caution"
assert classify_nws_heat_index(102.9) == "Extreme Caution"
assert classify_nws_heat_index(103.0) == "Danger"
assert classify_nws_heat_index(124.9) == "Danger"
assert classify_nws_heat_index(125.0) == "Extreme Danger"

assert evaluate_project_overnight_recovery(5.99) is False
assert evaluate_project_overnight_recovery(6.0) is True

print("Boundary checks passed.")

Boundary checks passed.


In [15]:
validation_results = []

for _, row in df_vectors.iterrows():
    actual_category = classify_nws_heat_index(row["heat_index_f"])
    actual_recovery = evaluate_project_overnight_recovery(
        row["hours_below_80f"]
    )

    category_pass = (
        actual_category == row["expected_nws_category"]
    )

    recovery_pass = (
        actual_recovery == row["expected_recovery_indicator"]
    )

    overall_pass = category_pass and recovery_pass

    validation_results.append(
        {
            "test_id": row["test_id"],
            "heat_index_f": row["heat_index_f"],
            "hours_below_80f": row["hours_below_80f"],
            "expected_category": row["expected_nws_category"],
            "actual_category": actual_category,
            "category_pass": category_pass,
            "expected_recovery": row["expected_recovery_indicator"],
            "actual_recovery": actual_recovery,
            "recovery_pass": recovery_pass,
            "overall_pass": overall_pass,
        }
    )

df_validation = pd.DataFrame(validation_results)

display(df_validation)

,test_id,heat_index_f,hours_below_80f,expected_category,actual_category,category_pass,expected_recovery,actual_recovery,recovery_pass,overall_pass
0,TC-001,78.0,7.0,Below Caution,Below Caution,True,True,True,True,True
1,TC-002,85.0,5.0,Caution,Caution,True,False,False,True,True
2,TC-003,98.0,4.0,Extreme Caution,Extreme Caution,True,False,False,True,True
3,TC-004,110.0,2.0,Danger,Danger,True,False,False,True,True
4,TC-005,128.0,0.0,Extreme Danger,Extreme Danger,True,False,False,True,True


In [16]:
total_tests = len(df_validation)
passed_tests = int(df_validation["overall_pass"].sum())
failed_tests = total_tests - passed_tests

print("=== Heat Science Validation Result ===")
print(f"Total tests : {total_tests}")
print(f"Passed      : {passed_tests}")
print(f"Failed      : {failed_tests}")

if failed_tests == 0:
    print("\n✅ All test vectors passed.")
else:
    print("\n❌ Some test vectors failed.")

=== Heat Science Validation Result ===
Total tests : 5
Passed      : 5
Failed      : 0

✅ All test vectors passed.


In [17]:
assert failed_tests == 0, (
    f"{failed_tests} test vector(s) failed. "
    "Fix the implementation or expected values before release."
)

print("✅ Validation suite is clean.")

✅ Validation suite is clean.


In [18]:
shared_test_vectors = df_vectors.copy()

shared_test_vectors["notes"] = [
    "Below 80°F: project-defined label below NWS Caution threshold.",
    "NWS Caution range.",
    "NWS Extreme Caution range.",
    "NWS Danger range.",
    "NWS Extreme Danger range.",
]

display(shared_test_vectors)

,test_id,heat_index_f,hours_below_80f,expected_nws_category,expected_recovery_indicator,notes
0,TC-001,78.0,7.0,Below Caution,True,Below 80°F: project-defined label below NWS Ca...
1,TC-002,85.0,5.0,Caution,False,NWS Caution range.
2,TC-003,98.0,4.0,Extreme Caution,False,NWS Extreme Caution range.
3,TC-004,110.0,2.0,Danger,False,NWS Danger range.
4,TC-005,128.0,0.0,Extreme Danger,False,NWS Extreme Danger range.


In [19]:
shared_test_vectors.to_csv(
    CSV_PATH,
    index=False
)

print(f"✅ Shared test vectors saved to:\n{CSV_PATH.resolve()}")

✅ Shared test vectors saved to:
C:\Users\nasse\temperature-api-quickstart\test-vectors\heat_science_test_vectors.csv


In [20]:
summary = {
    "total_test_cases": total_tests,
    "passed": passed_tests,
    "failed": failed_tests,
    "validation_status": "PASS" if failed_tests == 0 else "FAIL",
    "recovery_threshold_f": RECOVERY_THRESHOLD_F,
    "recovery_min_hours": RECOVERY_MIN_HOURS,
}

pd.Series(summary)

total_test_cases           5
passed                     5
failed                     0
validation_status       PASS
recovery_threshold_f    80.0
recovery_min_hours       6.0
dtype: object